In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"dataset\pollution.csv")

In [3]:
df.head()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,"Lat,Lon",Status
0,E20000,PL-08,2026-01-01 00:00+05:30,73.1,18.7,25.4,ug/m³,"13.2214,81.4807",Ok
1,E20001,PL-01,2026-01-01 00:10,73.1,26.2,49.9,ug/m3,"12.6208,79.5575",ok
2,E20002,PL-10,2026-01-01 00:20,83.0,26.1,45.5,UG/m3,"12.727,81.2451",MAINTENANCE
3,E20003,PL-17,2026-01-01 00:30,64.5,31.0,49.9,ug/m3,"13.5187,81.0188",Maint
4,E20004,PL-11,2026-01-01 00:40,82.1,22.0,51.9,µg/m³,"13.2038,80.0315",OK


In [4]:
df.tail()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,"Lat,Lon",Status
6625,E22690,PL-01,2026-01-19 16:20,5.7,22.8,24.7,mg/nm3,"12.6203,79.9658",FAULT
6626,E26085,PL-14,2026-02-12 06:10,129.6,35.8,80.4,UG/m3,"13.4076,80.4396",maintenance
6627,E20397,PL-02,2026-01-03 18:10,11.0,NaN,11.0,UG/m3,"13.6809,79.516",ok
6628,E24722,PL-02,2026-02-02 19:00,17.1,9.0,24.2,mg/Nm3,"14.4605,80.7726",Maint
6629,E21263,PL-20,2026-01-09 18:30,27.4,NaN,31.5,mg/nm3,"13.7189,81.1412",OK


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6630 entries, 0 to 6629
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Record_ID  6630 non-null   str    
 1   Plant_ID   6630 non-null   str    
 2   TS         6630 non-null   str    
 3   PM2.5      6549 non-null   float64
 4   SO2        6286 non-null   float64
 5   NOx        6433 non-null   float64
 6   Unit       6630 non-null   str    
 7   Lat,Lon    6559 non-null   str    
 8   Status     6612 non-null   str    
dtypes: float64(3), str(6)
memory usage: 466.3 KB


In [6]:
df.describe()

,PM2.5,SO2,NOx
count,6549.000000,6286.000000,6433.000000
mean,141.319386,127.904637,152.930365
std,1457.528654,1751.110746,1902.598084
min,-1041.500000,-49.830000,-1276.000000
25%,36.400000,18.300000,28.100000
50%,66.900000,25.200000,48.600000
75%,99.100000,32.000000,69.700000
max,41504.200000,48937.700000,49329.000000


In [7]:
demo = df
demo.columns = df.columns.str.strip()
demo.head()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,"Lat,Lon",Status
0,E20000,PL-08,2026-01-01 00:00+05:30,73.1,18.7,25.4,ug/m³,"13.2214,81.4807",Ok
1,E20001,PL-01,2026-01-01 00:10,73.1,26.2,49.9,ug/m3,"12.6208,79.5575",ok
2,E20002,PL-10,2026-01-01 00:20,83.0,26.1,45.5,UG/m3,"12.727,81.2451",MAINTENANCE
3,E20003,PL-17,2026-01-01 00:30,64.5,31.0,49.9,ug/m3,"13.5187,81.0188",Maint
4,E20004,PL-11,2026-01-01 00:40,82.1,22.0,51.9,µg/m³,"13.2038,80.0315",OK


---------- 1. Datetime normalization (24:10 → next day 00:10) ----------

In [8]:
def fix_datetime(ts):
    ts = str(ts)

    date_part = ts.split()[0]
    time_part = ts.split()[1]
    time_part = time_part.split("+")[0]

    hour = int(time_part.split(":")[0])
    minute = int(time_part.split(":")[1])

    date = pd.to_datetime(date_part, dayfirst=True)

    extra_days = hour // 24
    new_hour = hour % 24

    date = date + pd.Timedelta(days=extra_days)

    return pd.to_datetime(f"{date.date()} {new_hour:02d}:{minute:02d}")

demo["TS"] = demo["TS"].apply(fix_datetime)
demo.head()

C:\Users\KIIT\AppData\Local\Temp\ipykernel_11648\1189334576.py:11: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  date = pd.to_datetime(date_part, dayfirst=True)


,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,"Lat,Lon",Status
0,E20000,PL-08,2026-01-01 00:00:00,73.1,18.7,25.4,ug/m³,"13.2214,81.4807",Ok
1,E20001,PL-01,2026-01-01 00:10:00,73.1,26.2,49.9,ug/m3,"12.6208,79.5575",ok
2,E20002,PL-10,2026-01-01 00:20:00,83.0,26.1,45.5,UG/m3,"12.727,81.2451",MAINTENANCE
3,E20003,PL-17,2026-01-01 00:30:00,64.5,31.0,49.9,ug/m3,"13.5187,81.0188",Maint
4,E20004,PL-11,2026-01-01 00:40:00,82.1,22.0,51.9,µg/m³,"13.2038,80.0315",OK


---------- 2. Unit standardization (ug/m3, µg/m³ → ug/m3) ----------

In [9]:
demo["Unit"] = (demo["Unit"].astype(str).str.strip().str.lower().replace({
        "ug/m³": "ug/m3",
        "µg/m³": "ug/m3",
        "ug/m3": "ug/m3",
        "ug/m3 ": "ug/m3",
        "mg/nm3": "mg/nm3",
        "mg/nm³": "mg/nm3"})
)

demo["Unit"].unique()

<StringArray>
['ug/m3', 'mg/nm3']
Length: 2, dtype: str

---------- 3. Whitespace trimming and casing standardization for status ----------

In [10]:
demo["Status"] = (demo["Status"].astype(str).str.strip().str.upper())
demo["Status"] = demo["Status"].fillna("unknown")
demo["Status"] = (demo["Status"].astype(str).str.strip().str.lower().replace({
        "maint": "maintenance",
        "maintenance": "maintenance",
        "err": "fault",
        "fault": "fault",
        "--": "unknown",
        "0": "unknown",
        "?": "unknown",
        "nan": "unknown"})
)

demo["Status"].unique()

<StringArray>
['ok', 'maintenance', 'fault', 'unknown']
Length: 4, dtype: str

---------- 4. Coordinate parsing and range validation ----------

In [11]:
demo[["Latitude", "Longitude"]] = demo["Lat,Lon"].str.split(",", expand=True)

demo["Latitude"] = pd.to_numeric(demo["Latitude"], errors="coerce")
demo["Longitude"] = pd.to_numeric(demo["Longitude"], errors="coerce")

demo.loc[(demo["Latitude"] < -90) | (demo["Latitude"] > 90), "Latitude"] = None
demo.loc[(demo["Longitude"] < -180) | (demo["Longitude"] > 180), "Longitude"] = None

demo = demo.drop(columns=["Lat,Lon"])
demo[["Latitude", "Longitude"]].head()

,Latitude,Longitude
0,13.2214,81.4807
1,12.6208,79.5575
2,12.7270,81.2451
3,13.5187,81.0188
4,13.2038,80.0315


---------- 5. Negative and impossible concentration checks ----------

In [12]:
demo["PM2.5"] = pd.to_numeric(demo["PM2.5"], errors="coerce")
demo["SO2"] = pd.to_numeric(demo["SO2"], errors="coerce")
demo["NOx"] = pd.to_numeric(demo["NOx"], errors="coerce")

demo.loc[demo["PM2.5"] < 0, "PM2.5"] = None
demo.loc[demo["SO2"] < 0, "SO2"] = None
demo.loc[demo["NOx"] < 0, "NOx"] = None

demo.loc[demo["PM2.5"] > 1000, "PM2.5"] = None
demo.loc[demo["SO2"] > 2000, "SO2"] = None
demo.loc[demo["NOx"] > 2000, "NOx"] = None

demo[["PM2.5", "SO2", "NOx"]].head()

,PM2.5,SO2,NOx
0,73.1,18.7,25.4
1,73.1,26.2,49.9
2,83.0,26.1,45.5
3,64.5,31.0,49.9
4,82.1,22.0,51.9


---------- 6. Duplicate merge by (Plant_ID, TS) ----------

In [13]:
demo[demo.duplicated(subset=["Plant_ID", "TS"], keep=False)]

demo["PM2.5"] = demo.groupby(["Plant_ID", "TS"])["PM2.5"].transform("mean")
demo["SO2"] = demo.groupby(["Plant_ID", "TS"])["SO2"].transform("mean")
demo["NOx"] = demo.groupby(["Plant_ID", "TS"])["NOx"].transform("mean")

---------- 7. Sensor span/zero calibration application ----------

In [14]:
calibration = {
    "PM2.5": {"zero": 2, "span": 1.05},
    "SO2": {"zero": 1, "span": 1.02},
    "NOx": {"zero": 1, "span": 1.03}
}

for i in calibration:
    zero = calibration[i]["zero"]
    span = calibration[i]["span"]

    demo[i] = (demo[i] - zero) * span

demo.head()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,Status,Latitude,Longitude
0,E20000,PL-08,2026-01-01 00:00:00,74.655,18.054,25.132,ug/m3,ok,13.2214,81.4807
1,E20001,PL-01,2026-01-01 00:10:00,74.655,25.704,50.367,ug/m3,ok,12.6208,79.5575
2,E20002,PL-10,2026-01-01 00:20:00,85.050,25.602,45.835,ug/m3,maintenance,12.7270,81.2451
3,E20003,PL-17,2026-01-01 00:30:00,65.625,30.600,50.367,ug/m3,maintenance,13.5187,81.0188
4,E20004,PL-11,2026-01-01 00:40:00,84.105,21.420,52.427,ug/m3,ok,13.2038,80.0315


---------- 8. Outlier filtering using rolling median and Hampel filters ----------

In [15]:
def hampel_filter(series, w=5, n=3):
    rolling_median = series.rolling(w).median()
    diff = np.abs(series - rolling_median)
    mad = diff.rolling(w).median()

    threshold = n * mad
    outlier = diff > threshold

    series[outlier] = np.nan
    return series

demo["PM2.5"] = hampel_filter(demo["PM2.5"])
demo["SO2"] = hampel_filter(demo["SO2"])
demo["NOx"] = hampel_filter(demo["NOx"])

demo.head()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,Status,Latitude,Longitude
0,E20000,PL-08,2026-01-01 00:00:00,74.655,18.054,25.132,ug/m3,ok,13.2214,81.4807
1,E20001,PL-01,2026-01-01 00:10:00,74.655,25.704,50.367,ug/m3,ok,12.6208,79.5575
2,E20002,PL-10,2026-01-01 00:20:00,85.050,25.602,45.835,ug/m3,maintenance,12.7270,81.2451
3,E20003,PL-17,2026-01-01 00:30:00,65.625,30.600,50.367,ug/m3,maintenance,13.5187,81.0188
4,E20004,PL-11,2026-01-01 00:40:00,84.105,21.420,52.427,ug/m3,ok,13.2038,80.0315


---------- 9. Missing gap filling within allowed substitution rules ----------

In [16]:
pollutants = ["PM2.5", "SO2", "NOx"]

demo[pollutants] = demo[pollutants].interpolate(limit=2)
demo[pollutants].isna().sum()

demo.head()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,Status,Latitude,Longitude
0,E20000,PL-08,2026-01-01 00:00:00,74.655,18.054,25.132,ug/m3,ok,13.2214,81.4807
1,E20001,PL-01,2026-01-01 00:10:00,74.655,25.704,50.367,ug/m3,ok,12.6208,79.5575
2,E20002,PL-10,2026-01-01 00:20:00,85.050,25.602,45.835,ug/m3,maintenance,12.7270,81.2451
3,E20003,PL-17,2026-01-01 00:30:00,65.625,30.600,50.367,ug/m3,maintenance,13.5187,81.0188
4,E20004,PL-11,2026-01-01 00:40:00,84.105,21.420,52.427,ug/m3,ok,13.2038,80.0315


---------- 10. Instrument downtime labeling from maintenance logs ----------

In [17]:
demo["Downtime"] = demo["Status"].isin(["maintenance"])
demo[["Status", "Downtime"]].head()

,Status,Downtime
0,ok,False
1,ok,False
2,maintenance,True
3,maintenance,True
4,ok,False


---------- 11. Limit of detection (LOD) handling (below LOD → half LOD) ----------

In [18]:
LOD = {"PM2.5": 5, "SO2": 3, "NOx": 4}

demo["LOD_Flag"] = False
for col,lod in LOD.items():
    demo[col] = demo[col].astype(str).str.replace("<", "", regex=False)
    demo[col] = pd.to_numeric(demo[col], errors="coerce")

    mask = demo[col] < lod
    demo.loc[mask, col] = lod / 2
    demo.loc[mask, "LOD_Flag"] = True

demo[["PM2.5", "SO2", "NOx", "LOD_Flag"]].head()

,PM2.5,SO2,NOx,LOD_Flag
0,74.655,18.054,25.132,False
1,74.655,25.704,50.367,False
2,85.050,25.602,45.835,False
3,65.625,30.600,50.367,False
4,84.105,21.420,52.427,False


---------- 12. STATUS NORMALIZATION ----------

In [19]:
demo["Status"] = demo["Status"].astype(str).str.strip().str.upper()

status_map ={
    "OK": "OK",
    "NORMAL": "OK",
    "RUNNING": "OK",
    "FAULT": "FAULT",
    "ERROR": "FAULT",
    "FAIL": "FAULT",
    "MAINT": "MAINT",
    "MAINTENANCE": "MAINT",
    "SERVICE": "MAINT"
}

demo["Status"] = demo["Status"].replace(status_map)
valid_status = ["OK", "FAULT", "MAINT"]
demo["Unknown_Status_Flag"] = ~demo["Status"].isin(valid_status)
demo.loc[demo["Unknown_Status_Flag"], "Status"] = "UNKNOWN"
demo.loc[demo["Unknown_Status_Flag"], "Status"].value_counts()

Status
UNKNOWN    60
Name: count, dtype: int64

---------- 13. STACK TO PLANT MAPPING VALIDATION ----------

In [20]:
plant_registry = pd.read_csv(r"dataset\plant_stack_master.csv")
plant_registry.head()

,Plant_ID,Stack_ID,Plant_Name,Plant_Location,State,Stack_Height_m,Capacity_MW,Fuel_Type,Commission_Year,Status
0,PL-01,STK-001-A,North Thermal Plant,Delhi,Delhi,210,500,Coal,2012,Active
1,PL-02,STK-002-A,South Power Station,Chennai,Tamil Nadu,220,600,Coal,2015,Active
2,PL-03,STK-003-A,Western Energy Plant,Mumbai,Maharashtra,200,450,Gas,2018,Active
3,PL-04,STK-004-A,Eastern Thermal Plant,Kolkata,West Bengal,230,550,Coal,2010,Active
4,PL-05,STK-005-A,Central Energy Station,Nagpur,Maharashtra,215,480,Coal,2014,Active


In [21]:
valid_plants = plant_registry["Plant_ID"].unique()
demo["Invalid_Plant_Flag"] = ~demo["Plant_ID"].isin(valid_plants)
demo[demo["Invalid_Plant_Flag"]].head()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,Status,Latitude,Longitude,Downtime,LOD_Flag,Unknown_Status_Flag,Invalid_Plant_Flag


---------- 14. Unit Conversion ----------

In [22]:
demo["Unit"] = demo["Unit"].str.strip().str.lower()
mask = demo["Unit"].isin(["mg/nm3", "mg/nm³"])
mask.sum()

np.int64(2297)

In [23]:
pollutants = ["PM2.5", "SO2", "NOx"]
demo.loc[mask, pollutants] = demo.loc[mask, pollutants] * 1000
demo.loc[mask, "Unit"] = "ug/m3"

demo["Unit_Converted_Flag"] = mask
demo["Unit"].value_counts()

Unit
ug/m3    6630
Name: count, dtype: int64

In [24]:
demo[demo["Unit_Converted_Flag"]].head()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,Status,Latitude,Longitude,Downtime,LOD_Flag,Unknown_Status_Flag,Invalid_Plant_Flag,Unit_Converted_Flag
6,E20006,PL-10,2026-01-01 01:00:00,46830.0,34680.0,668573.0,ug/m3,MAINT,13.1272,81.2341,True,False,False,False,True
7,E20007,PL-05,2026-01-01 01:10:00,76230.0,27132.0,61285.0,ug/m3,MAINT,12.9011,80.1421,True,False,False,False,True
9,E20009,PL-20,2026-01-01 01:30:00,84315.0,33864.0,61388.0,ug/m3,OK,13.5886,79.5058,False,False,False,False,True
13,E20013,PL-05,2026-01-01 02:10:00,64575.0,28050.0,58195.0,ug/m3,FAULT,13.6432,79.5181,False,False,False,False,True
14,E20014,PL-10,2026-01-01 02:20:00,95235.0,28288.0,66538.0,ug/m3,OK,14.0950,79.2735,False,False,False,False,True


In [25]:
demo.head()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,Status,Latitude,Longitude,Downtime,LOD_Flag,Unknown_Status_Flag,Invalid_Plant_Flag,Unit_Converted_Flag
0,E20000,PL-08,2026-01-01 00:00:00,74.655,18.054,25.132,ug/m3,OK,13.2214,81.4807,False,False,False,False,False
1,E20001,PL-01,2026-01-01 00:10:00,74.655,25.704,50.367,ug/m3,OK,12.6208,79.5575,False,False,False,False,False
2,E20002,PL-10,2026-01-01 00:20:00,85.050,25.602,45.835,ug/m3,MAINT,12.7270,81.2451,True,False,False,False,False
3,E20003,PL-17,2026-01-01 00:30:00,65.625,30.600,50.367,ug/m3,MAINT,13.5187,81.0188,True,False,False,False,False
4,E20004,PL-11,2026-01-01 00:40:00,84.105,21.420,52.427,ug/m3,OK,13.2038,80.0315,False,False,False,False,False


---------- 15. Ambient vs stack data source tagging ----------

In [26]:
sensor_registry = pd.read_csv(r"dataset\sensor_registry.csv")

demo = demo.merge(
    sensor_registry[["Plant_ID","Source_Type"]],
    on="Plant_ID",
    how="left"
)

demo["Source_Type"] = demo["Source_Type"].fillna("UNKNOWN")

demo[["Plant_ID","Source_Type"]].head()
demo["Source_Type"].value_counts()

Source_Type
STACK      2683
AMBIENT    2324
UNKNOWN    1623
Name: count, dtype: int64

---------- 16. Lab result digitization QC (double entry checks) ----------

In [27]:
dup_check = demo.groupby(["Plant_ID","TS"]).agg({
    "PM2.5":"nunique",
    "SO2":"nunique",
    "NOx":"nunique"
}).reset_index()

dup_check["Digitization_Issue"] = (
    (dup_check["PM2.5"] > 1) |
    (dup_check["SO2"] > 1) |
    (dup_check["NOx"] > 1)
)

dup_check[dup_check["Digitization_Issue"]].head()

,Plant_ID,TS,PM2.5,SO2,NOx,Digitization_Issue
74,PL-01,2026-01-19 16:20:00,1,1,2,True
174,PL-01,2026-02-02 20:40:00,2,2,1,True
181,PL-01,2026-02-07 12:50:00,2,1,1,True
185,PL-01,2026-02-13 10:40:00,2,1,1,True
226,PL-01,2026-05-01 04:20:00,1,2,1,True


In [28]:
dup_check["Digitization_Issue"].sum()

np.int64(81)

In [29]:
problem_keys = dup_check[dup_check["Digitization_Issue"]][["Plant_ID","TS"]]

demo.merge(problem_keys, on=["Plant_ID","TS"]).head()

,Record_ID,Plant_ID,TS,PM2.5,SO2,NOx,Unit,Status,Latitude,Longitude,Downtime,LOD_Flag,Unknown_Status_Flag,Invalid_Plant_Flag,Unit_Converted_Flag,Source_Type
0,E20145,PL-08,2026-02-01 00:10:00,55.545,22.542,42.4360,ug/m3,MAINT,12.7781,79.6132,True,False,False,False,False,AMBIENT
1,E20185,PL-19,2026-02-01 06:50:00,108.605,32.691,93.0090,ug/m3,OK,12.8908,80.2995,False,False,False,False,False,UNKNOWN
2,E20397,PL-02,2026-03-01 18:10:00,9.240,7.242,10.5060,ug/m3,OK,13.6809,79.5160,False,False,False,False,False,STACK
3,E20506,PL-10,2026-01-05 00:20:00,47.565,22.185,49.7490,ug/m3,MAINT,13.3914,79.1510,True,False,False,False,False,STACK
4,E20560,PL-10,2026-01-05 00:20:00,47.565,22.185,28.2735,ug/m3,OK,12.5432,79.5182,False,False,False,False,False,STACK


---------- 17 — Standard Time Base Conversion to IST ----------

In [30]:
demo["TS"] = pd.to_datetime(demo["TS"], errors="coerce")
demo["TS_IST"] = demo["TS"].dt.tz_localize("UTC").dt.tz_convert("Asia/Kolkata")

demo[["TS","TS_IST"]].head()

,TS,TS_IST
0,2026-01-01 00:00:00,2026-01-01 05:30:00+05:30
1,2026-01-01 00:10:00,2026-01-01 05:40:00+05:30
2,2026-01-01 00:20:00,2026-01-01 05:50:00+05:30
3,2026-01-01 00:30:00,2026-01-01 06:00:00+05:30
4,2026-01-01 00:40:00,2026-01-01 06:10:00+05:30


---------- 18. PII removal in inspection notes ----------

In [31]:
import re

notes = [
    "Checked by Rahul Sharma contact 9876543210",
    "Maintenance by engineer raj@gmail.com",
    "Routine inspection completed"
] * (len(demo)//3 + 1)

demo["Inspection_Notes"] = notes[:len(demo)]

def remove_pii(text):
    text = str(text)
    
    text = re.sub(r"\b\d{10}\b", "[REDACTED]", text)
    text = re.sub(r"\S+@\S+", "[REDACTED]", text)
    
    return text

demo["Inspection_Notes"] = demo["Inspection_Notes"].apply(remove_pii)
demo[["Inspection_Notes"]].head()

,Inspection_Notes
0,Checked by Rahul Sharma contact [REDACTED]
1,Maintenance by engineer [REDACTED]
2,Routine inspection completed
3,Checked by Rahul Sharma contact [REDACTED]
4,Maintenance by engineer [REDACTED]


--------- 19. Load Regulatory Limits ----------

In [32]:
limits = pd.read_csv(r"dataset\regulatory_limits.csv")
limits

,Pollutant,Limit_ugm3,Averaging_Period,Source
0,PM2.5,60,24-hour,Example_Regulatory_Standard
1,SO2,80,24-hour,Example_Regulatory_Standard
2,NOx,80,24-hour,Example_Regulatory_Standard


In [33]:
limit_dict = dict(zip(limits["Pollutant"], limits["Limit_ugm3"]))
limit_dict

{'PM2.5': 60, 'SO2': 80, 'NOx': 80}

In [34]:
for col in pollutants:
    demo[col+"_Exceed"] = demo[col] > limit_dict[col]

demo[[ "PM2.5","PM2.5_Exceed",
       "SO2","SO2_Exceed",
       "NOx","NOx_Exceed"]].head()

,PM2.5,PM2.5_Exceed,SO2,SO2_Exceed,NOx,NOx_Exceed
0,74.655,True,18.054,False,25.132,False
1,74.655,True,25.704,False,50.367,False
2,85.050,True,25.602,False,45.835,False
3,65.625,True,30.600,False,50.367,False
4,84.105,True,21.420,False,52.427,False


In [35]:
demo[["PM2.5_Exceed","SO2_Exceed","NOx_Exceed"]].sum()

PM2.5_Exceed    4783
SO2_Exceed      2302
NOx_Exceed      2853
dtype: int64

--------- 20. Audit trail creation for corrections. ----------

In [45]:
sample = pd.DataFrame({
    "pm2.5": [120, -15, 180, -8],
    "so2": [-56, 76, -54, -67],
    "nox": [56, 76, -3, 33]
})

In [46]:
audit_log = []

In [47]:
pollutants = ["pm2.5", "so2", "nox"]

for col in pollutants:
    mask = sample[col] < 0
    
    for i in sample[mask].index:
        audit_log.append({
            "row_id": i,
            "column": col,
            "old_value": sample.loc[i, col],
            "new_value": None,
            "reason": "Negative concentration corrected"
        })
    
    sample.loc[mask, col] = None

audit_df = pd.DataFrame(audit_log)

print("Corrected Data:")
print(sample)

print("\nAudit Log:")
print(audit_df)

   row_id column  old_value new_value                            reason
0       1  pm2.5        -15      None  Negative concentration corrected
1       3  pm2.5         -8      None  Negative concentration corrected


In [39]:
demo = demo.rename(columns={
    "PM2.5": "PM2_5_ugm3",
    "SO2": "SO2_ugm3",
    "NOx": "NOx_ugm3"
})

demo.to_csv("cleaned_pollution.csv", index=False)
demo

,Record_ID,Plant_ID,TS,PM2_5_ugm3,SO2_ugm3,NOx_ugm3,Unit,Status,Latitude,Longitude,...,LOD_Flag,Unknown_Status_Flag,Invalid_Plant_Flag,Unit_Converted_Flag,Source_Type,TS_IST,Inspection_Notes,PM2.5_Exceed,SO2_Exceed,NOx_Exceed
0,E20000,PL-08,2026-01-01 00:00:00,74.655,18.054,25.132,ug/m3,OK,13.2214,81.4807,...,False,False,False,False,AMBIENT,2026-01-01 05:30:00+05:30,Checked by Rahul Sharma contact [REDACTED],True,False,False
1,E20001,PL-01,2026-01-01 00:10:00,74.655,25.704,50.367,ug/m3,OK,12.6208,79.5575,...,False,False,False,False,STACK,2026-01-01 05:40:00+05:30,Maintenance by engineer [REDACTED],True,False,False
2,E20002,PL-10,2026-01-01 00:20:00,85.050,25.602,45.835,ug/m3,MAINT,12.7270,81.2451,...,False,False,False,False,STACK,2026-01-01 05:50:00+05:30,Routine inspection completed,True,False,False
3,E20003,PL-17,2026-01-01 00:30:00,65.625,30.600,50.367,ug/m3,MAINT,13.5187,81.0188,...,False,False,False,False,UNKNOWN,2026-01-01 06:00:00+05:30,Checked by Rahul Sharma contact [REDACTED],True,False,False
4,E20004,PL-11,2026-01-01 00:40:00,84.105,21.420,52.427,ug/m3,OK,13.2038,80.0315,...,False,False,False,False,AMBIENT,2026-01-01 06:10:00+05:30,Maintenance by engineer [REDACTED],True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6625,E22690,PL-01,2026-01-19 16:20:00,2500.000,22185.000,85284.000,ug/m3,FAULT,12.6203,79.9658,...,True,False,False,True,STACK,2026-01-19 21:50:00+05:30,Maintenance by engineer [REDACTED],True,True,True
6626,E26085,PL-14,2026-12-02 06:10:00,131.880,34.731,82.709,ug/m3,MAINT,13.4076,80.4396,...,False,False,False,False,STACK,2026-12-02 11:40:00+05:30,Routine inspection completed,True,False,True
6627,E20397,PL-02,2026-03-01 18:10:00,9.240,7.242,53.148,ug/m3,OK,13.6809,79.5160,...,False,False,False,False,STACK,2026-03-01 23:40:00+05:30,Checked by Rahul Sharma contact [REDACTED],False,False,False
6628,E24722,PL-02,2026-02-02 19:00:00,16327.500,8109.000,23587.000,ug/m3,MAINT,14.4605,80.7726,...,False,False,False,True,STACK,2026-02-03 00:30:00+05:30,Maintenance by engineer [REDACTED],True,True,True
